# Simulación Monte Carlo - Clorificador
Proyecto listo para Visual Studio Code + Jupyter

In [ ]:
%pip install numpy pandas plotly kaleido -q

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path
import json

np.random.seed(42)
out = Path('output')
out.mkdir(exist_ok=True)

N = 1000
Q = np.clip(np.random.normal(1.0, 0.25, N), 0.2, None)
t = np.clip(np.random.normal(20, 4, N), 5, None)
C0 = np.clip(np.random.normal(1.2, 0.3, N), 0.2, None)
demand = np.clip(np.random.normal(0.65, 0.18, N), 0.1, None)
frac = np.clip(np.random.beta(2.5, 5.5, N), 0.05, 0.95)
k = np.clip(np.random.normal(0.035, 0.01, N), 0.005, None)
T = np.clip(np.random.normal(28, 3, N), 18, None)
pH = np.clip(np.random.normal(7.2, 0.4, N), 6.0, 8.5)
turb = np.clip(np.random.gamma(2.2, 0.6, N), 0.1, None)

T_factor = np.exp(0.03 * (T - 25))
pH_factor = 1 + 0.06 * np.abs(pH - 7.2)
turb_factor = 1 + 0.15 * turb
Q_factor = 1 + 0.08 * (Q - 1.0)

Cres = np.maximum((C0 * frac - demand) * np.exp(-k * t) / (T_factor * pH_factor) / turb_factor / Q_factor, 0)
status = np.where(Cres < 0.2, 'Subclorada', np.where(Cres <= 0.5, 'Adecuada', 'Sobreclorada'))

sim = pd.DataFrame({
    'Q_Ls': Q,
    'tiempo_min': t,
    'cloro_inicial_mgL': C0,
    'demanda_mgL': demand,
    'fraccion_dosificada': frac,
    'k_decay': k,
    'temperatura_C': T,
    'pH': pH,
    'turbidez_NTU': turb,
    'cloro_residual_mgL': Cres,
    'estado': status
})

sim.to_csv(out / 'simulacion_montecarlo_clorificador.csv', index=False)
eda = sim.describe(include='all').T
eda.to_csv(out / 'eda_montecarlo_clorificador.csv')

fig1 = px.histogram(sim, x='cloro_residual_mgL', nbins=35)
fig1.write_image(out / 'hist_cloro_residual.png')

fig2 = px.histogram(sim, x='estado', color='estado')
fig2.write_image(out / 'escenarios_simulados.png')

fig3 = px.box(sim, y='cloro_residual_mgL')
fig3.write_image(out / 'box_cloro_residual.png')

print("Proyecto ejecutado correctamente")
sim.head()
